# HEBI explorer on a free Colab GPU

This notebook starts the HEBI web page inside Colab. You use the page as usual: upload `.h5ad` sections, run HEBI zero-shot or train first, and explore clusters, niches and genes. Nothing to code.

**Steps**
1. Menu **Runtime → Change runtime type → T4 GPU**, then **Save**.
2. Menu **Runtime → Run all**.
3. Wait about 5 minutes. The HEBI page opens at the bottom of the notebook.

Your files stay in this Colab session and are gone when it ends. Code: [github.com/roxana9/hebi](https://github.com/roxana9/hebi) (MIT). Weights: [huggingface.co/Roxana9/hebi](https://huggingface.co/Roxana9/hebi) (CC BY-NC 4.0, non-commercial research use).

In [ ]:
#@title 1. Install HEBI (about 4 to 6 minutes)
# HEBI needs torch_sparse, which has prebuilt wheels only for some PyTorch
# versions. Colab's own PyTorch is often newer, so a matching PyTorch (2.8) is
# installed here. The web server runs in its own process, so no restart is needed.
import subprocess, sys, shutil
gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
cu = "cu126" if gpu else "cpu"
print("GPU:", "yes" if gpu else "no (zero-shot still works on CPU; for a GPU use Runtime → Change runtime type → T4 GPU)")
def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:]); raise SystemExit("installation failed, see the messages above")
pip("torch==2.8.0", "--index-url", f"https://download.pytorch.org/whl/{cu}")
pip("torch-scatter", "torch-sparse", "--only-binary", ":all:", "-f", f"https://data.pyg.org/whl/torch-2.8.0+{cu}.html")
pip("hebi @ git+https://github.com/roxana9/hebi@main", "fastapi", "uvicorn[standard]", "umap-learn", "huggingface_hub")
r = subprocess.run([sys.executable, "-c", "import torch, torch_sparse, hebi; print(torch.__version__, torch.cuda.is_available())"],
                   capture_output=True, text=True)
print("installed:", r.stdout.strip() or r.stderr[-2000:])

In [ ]:
#@title 2. Get the web page and the pretrained weights
import os, urllib.request
from huggingface_hub import hf_hub_download
WEB = "/content/hebi_web"; os.makedirs(WEB, exist_ok=True)
RAW = "https://raw.githubusercontent.com/roxana9/hebi/main/web"
for f in ["app.py", "index.html", "niche_merge.py"]:
    urllib.request.urlretrieve(f"{RAW}/{f}", f"{WEB}/{f}")
ckpt = hf_hub_download("Roxana9/hebi", "hebi_hybrid_oldL0.pt")
vocab = hf_hub_download("Roxana9/hebi", "default_gene_vocab.json")
# example sections offered on the page (no upload needed)
EXAMPLES = "/content/hebi_examples"; os.makedirs(EXAMPLES, exist_ok=True)
for f in ["xenium_breast_cancer_rep1.h5ad", "xenium_breast_cancer_rep2.h5ad"]:
    try:
        p = hf_hub_download("Roxana9/Hebi", f, repo_type="dataset")
        if not os.path.exists(f"{EXAMPLES}/{f}"): os.symlink(p, f"{EXAMPLES}/{f}")
    except Exception as e:
        print("example not available:", f, e)
print("ready:", ckpt)

In [ ]:
#@title 3. Start HEBI and open the page
import subprocess, time, json, urllib.request, os
env = dict(os.environ, HEBI_CHECKPOINT=ckpt, HEBI_VOCAB=vocab, HEBI_WEB_HOST="127.0.0.1", HEBI_WEB_PORT="8000",
           HEBI_WEB_WORKDIR="/content/hebi_jobs", HEBI_WEB_MAX_UPLOAD_GB="4", HEBI_WEB_MAX_EPOCHS="200", HEBI_WEB_EXAMPLES=EXAMPLES)
log = open("/content/hebi_web.log", "w")
server = subprocess.Popen([sys.executable, "app.py"], cwd=WEB, env=env, stdout=log, stderr=subprocess.STDOUT)
print("starting HEBI (loads PyTorch and the model, up to a few minutes)", end="")
for _ in range(120):
    time.sleep(5); print(".", end="")
    try:
        if json.load(urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=5)).get("ok"): break
    except Exception: pass
    if server.poll() is not None: break
print()
if server.poll() is not None:
    print(open("/content/hebi_web.log").read()[-3000:])
else:
    from google.colab import output
    print("HEBI is running. The page opens below; keep this tab open while you work.")
    output.serve_kernel_port_as_iframe(8000, path="/", height=900)